# Bronze JSON Explorer

Notebook para explorar solo los JSON de Bronze. Carga los archivos, los normaliza a tablas con pandas y deja una base para el análisis exploratorio.

In [24]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

def find_project_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / 'README.md').exists() and (candidate / 'datalake_bronze').exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root()
ENV_FILE = PROJECT_ROOT / 'airflow' / '.env'
DEFAULT_BRONZE_DIR = PROJECT_ROOT / 'datalake_bronze'

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'ENV_FILE = {ENV_FILE}')
print(f'DEFAULT_BRONZE_DIR = {DEFAULT_BRONZE_DIR}')

PROJECT_ROOT = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming
ENV_FILE = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/airflow/.env
DEFAULT_BRONZE_DIR = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_bronze


In [25]:
def load_env_file(path: Path) -> dict[str, str]:
    values: dict[str, str] = {}
    if not path.exists():
        return values
    for raw_line in path.read_text(encoding='utf-8').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


def resolve_bronze_dir(raw_value: str | Path | None) -> Path:
    if raw_value is None:
        return DEFAULT_BRONZE_DIR
    candidate = Path(raw_value)
    if not candidate.is_absolute():
        candidate = (PROJECT_ROOT / candidate).resolve()
    if candidate.exists():
        return candidate
    return DEFAULT_BRONZE_DIR


env = load_env_file(ENV_FILE)
bronze_dir = resolve_bronze_dir(env.get('BRONZE_BASE_PATH'))
bronze_files = sorted(bronze_dir.glob('*.json'))
print(f'BRONZE_DIR = {bronze_dir}')
print(f'JSON files found = {len(bronze_files)}')
display(pd.DataFrame({'bronze_json': [path.name for path in bronze_files]}))

BRONZE_DIR = /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/datalake_bronze
JSON files found = 7


,bronze_json
0,twitter_tweets_20260411_024745.json
1,twitter_tweets_20260524_202328.json
2,twitter_tweets_20260524_202549.json
3,twitter_tweets_20260524_202604.json
4,webscraping_noticias_20260411_024656.json
5,webscraping_noticias_20260524_202305.json
6,webscraping_noticias_20260524_202602.json


## Funciones de normalización

Estas funciones convierten cada JSON Bronze en tablas de pandas más fáciles de leer y analizar.

In [26]:
def load_json_payload(path: Path):
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def infer_source(path: Path) -> str:
    name = path.name.lower()
    if 'webscraping' in name:
        return 'webscraping'
    if 'twitter' in name:
        return 'twitter'
    return 'unknown'


def ensure_list(payload):
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        return [payload]
    return []


def build_webscraping_tables(path: Path) -> dict[str, pd.DataFrame]:
    payload = ensure_list(load_json_payload(path))
    snapshots = pd.json_normalize(payload, sep='_')
    snapshots['bronze_file'] = path.name
    if 'date' in snapshots.columns:
        snapshots['date_parsed'] = pd.to_datetime(pd.to_numeric(snapshots['date'], errors='coerce'), unit='ms', errors='coerce')
    if 'createdAt' in snapshots.columns:
        snapshots['createdAt'] = pd.to_datetime(snapshots['createdAt'], errors='coerce')
    if 'updatedAt' in snapshots.columns:
        snapshots['updatedAt'] = pd.to_datetime(snapshots['updatedAt'], errors='coerce')
    if 'news' not in snapshots.columns:
        return {'webscraping_snapshots': snapshots}

    snapshots['news_count'] = snapshots['news'].apply(lambda value: len(value) if isinstance(value, list) else 0)
    news_rows = snapshots.explode('news', ignore_index=True)
    news_details = pd.json_normalize(news_rows['news']).add_prefix('news_')
    news_table = pd.concat([news_rows.drop(columns=['news']), news_details], axis=1)
    if 'news_comments' in news_table.columns:
        news_table['news_comments_count'] = news_table['news_comments'].apply(lambda value: len(value) if isinstance(value, list) else 0)
        comments_table = news_table.explode('news_comments', ignore_index=True).rename(columns={'news_comments': 'comment_text'})
        comments_table['comment_text'] = comments_table['comment_text'].astype('string')
    else:
        comments_table = news_table.copy()

    return {
        'webscraping_snapshots': snapshots.drop(columns=['news']),
        'webscraping_news': news_table,
        'webscraping_comments': comments_table,
    }


def build_twitter_tables(path: Path) -> dict[str, pd.DataFrame]:
    payload = ensure_list(load_json_payload(path))
    tweets = pd.json_normalize(payload, sep='_')
    tweets['bronze_file'] = path.name
    if 'createdAt' in tweets.columns:
        tweets['createdAt'] = pd.to_datetime(tweets['createdAt'], errors='coerce')
    if 'text' in tweets.columns:
        tweets['text_length'] = tweets['text'].fillna('').astype(str).str.len()
    return {'twitter_tweets': tweets}

In [27]:
bronze_tables: dict[str, list[pd.DataFrame]] = {}

for bronze_file in bronze_files:
    source = infer_source(bronze_file)
    if source == 'webscraping':
        tables = build_webscraping_tables(bronze_file)
    elif source == 'twitter':
        tables = build_twitter_tables(bronze_file)
    else:
        continue

    for table_name, table in tables.items():
        bronze_tables.setdefault(table_name, []).append(table)

bronze_tables = {table_name: pd.concat(parts, ignore_index=True) for table_name, parts in bronze_tables.items()}

for table_name, table in bronze_tables.items():
    print(f'{table_name}: {table.shape[0]} rows x {table.shape[1]} columns')

twitter_tweets: 22 rows x 8 columns
webscraping_snapshots: 3 rows x 8 columns
webscraping_news: 30 rows x 12 columns
webscraping_comments: 589 rows x 12 columns


## Vista rápida de las tablas

Aquí revisamos columnas, tipos, valores faltantes y unas primeras filas por cada tabla.

In [28]:
for table_name, table in bronze_tables.items():
    display(Markdown(f'### {table_name}'))
    display(table.head(5))
    display(table.dtypes.astype(str).to_frame('dtype'))
    missing = table.isna().mean().sort_values(ascending=False).to_frame('missing_ratio')
    display(missing.head(15))

### twitter_tweets

,_id,tweets,has_next_page,next_cursor,createdAt,updatedAt,__v,bronze_file
0,69d86f08aae727ea3c009a31,"[{'type': 'tweet', 'id': '2042444537281593777'...",True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2026-04-10 03:31:20.837,2026-04-10T03:31:20.837000,0,twitter_tweets_20260411_024745.json
1,69d86f08aae727ea3c009a31,"[{'type': 'tweet', 'id': '2042444537281593777'...",True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2026-04-10 03:31:20.837,2026-04-10T03:31:20.837000,0,twitter_tweets_20260524_202328.json
2,69dcda8ac500fb828bc46659,"[{'type': 'tweet', 'id': '2043651774498640291'...",True,DAADDAABCgABHFyCdg-WkaMKAAIcTmzHWtpg4gAIAAIAAA...,2026-04-13 11:59:06.056,2026-04-13T11:59:06.056000,0,twitter_tweets_20260524_202328.json
3,69e0cda162226a6d2a189619,"[{'type': 'tweet', 'id': '2044702071639724279'...",True,DAADDAABCgABHGA9s2uXUPcKAAIcVjS1vxYx8gAIAAIAAA...,2026-04-16 11:53:05.918,2026-04-16T11:53:05.918000,0,twitter_tweets_20260524_202328.json
4,69e61524be825dd036d725ce,"[{'type': 'tweet', 'id': '2045741484456972706'...",True,DAADDAABCgABHGPvCpNWkaIKAAIcW3Pwx5aB8QAIAAIAAA...,2026-04-20 11:59:32.174,2026-04-20T11:59:32.174000,0,twitter_tweets_20260524_202328.json


,dtype
_id,str
tweets,object
has_next_page,bool
next_cursor,str
createdAt,datetime64[us]
updatedAt,str
__v,int64
bronze_file,str


,missing_ratio
next_cursor,0.409091
_id,0.000000
tweets,0.000000
has_next_page,0.000000
createdAt,0.000000
updatedAt,0.000000
__v,0.000000
bronze_file,0.000000


### webscraping_snapshots

...

,missing_ratio
_id,0.0
date,0.0
createdAt,0.0
updatedAt,0.0
__v,0.0
bronze_file,0.0
date_parsed,0.0
news_count,0.0
news_newsLink,0.0
news_comments,0.0


### webscraping_comments

,_id,date,createdAt,updatedAt,__v,bronze_file,date_parsed,news_count,news_newsLink,comment_text,news__id,news_comments_count
0,69d947bb4dd967352a8ee984,1775847355402,2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,webscraping_noticias_20260411_024656.json,2026-04-10 18:55:55.402,12,https://www.gsmarena.com/samsung_galaxy_fold8_...,"DarlingYext, 09 Apr 2026Oh no, the Samsung Fai...",69d947bb4dd967352a8ee985,11
1,69d947bb4dd967352a8ee984,1775847355402,2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,webscraping_noticias_20260411_024656.json,2026-04-10 18:55:55.402,12,https://www.gsmarena.com/samsung_galaxy_fold8_...,"DarlingYext, 09 Apr 2026Oh no, the Samsung Fai...",69d947bb4dd967352a8ee985,11
2,69d947bb4dd967352a8ee984,1775847355402,2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,webscraping_noticias_20260411_024656.json,2026-04-10 18:55:55.402,12,https://www.gsmarena.com/samsung_galaxy_fold8_...,epal fans say why copy ifone fold,69d947bb4dd967352a8ee985,11
3,69d947bb4dd967352a8ee984,1775847355402,2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,webscraping_noticias_20260411_024656.json,2026-04-10 18:55:55.402,12,https://www.gsmarena.com/samsung_galaxy_fold8_...,"let me guess, these dummies are STILL going to...",69d947bb4dd967352a8ee985,11
4,69d947bb4dd967352a8ee984,1775847355402,2026-04-10 18:55:55.421,2026-04-10 18:55:55.421,0,webscraping_noticias_20260411_024656.json,2026-04-10 18:55:55.402,12,https://www.gsmarena.com/samsung_galaxy_fold8_...,This is a confirmation that the Apple's upcomi...,69d947bb4dd967352a8ee985,11


,dtype
_id,str
date,str
createdAt,datetime64[us]
updatedAt,datetime64[us]
__v,int64
bronze_file,str
date_parsed,datetime64[ms]
news_count,int64
news_newsLink,str
comment_text,string


,missing_ratio
_id,0.0
date,0.0
createdAt,0.0
updatedAt,0.0
__v,0.0
bronze_file,0.0
date_parsed,0.0
news_count,0.0
news_newsLink,0.0
comment_text,0.0


## Análisis inicial

Estas celdas dejan listas algunas agregaciones básicas para empezar el análisis con pandas.

In [29]:
if 'webscraping_comments' in bronze_tables:
    comments = bronze_tables['webscraping_comments'].copy()
    if 'date_parsed' in comments.columns:
        comments['day'] = comments['date_parsed'].dt.date
        display(comments.groupby('day').size().sort_values(ascending=False).head(10).to_frame('comments'))
    if 'news_newsLink' in comments.columns:
        display(comments['news_newsLink'].value_counts().head(10).to_frame('comments'))

if 'twitter_tweets' in bronze_tables:
    tweets = bronze_tables['twitter_tweets'].copy()
    if 'createdAt' in tweets.columns:
        tweets['day'] = tweets['createdAt'].dt.date
        display(tweets.groupby('day').size().sort_values(ascending=False).head(10).to_frame('tweets'))
    if 'author_name' in tweets.columns:
        display(tweets['author_name'].value_counts().head(10).to_frame('tweets'))

,comments
day,
2026-05-24,352
2026-04-10,237


,comments
news_newsLink,
https://www.gsmarena.com/weekly_poll_results_sony_xperia_1_viii_might_be_a_hit_despite_its_high_price-news-72956.php,116
https://www.gsmarena.com/trump_mobile_confirms_that_it_exposed_customer_data-news-72963.php,110
https://www.gsmarena.com/oppo_find_x9_ultra_full_camera_specs_confirmed_design_leaked-news-72315.php,84
https://www.gsmarena.com/the_future_may_not_be_very_bright_for_the_oppo_find_x10_pro_max-news-72954.php,40
https://www.gsmarena.com/samsung_is_boosting_galaxy_s26_and_galaxy_s26_ultra_production-news-72318.php,38
https://www.gsmarena.com/xiaomi_17t_series_telephoto_camera_magnification-news-72968.php,34
https://www.gsmarena.com/honors_new_video_teaser_for_the_600_and_600_pro_showcases_their_design-news-72312.php,19
https://www.gsmarena.com/ai_unveils_nova_2_series_and_nova_flip_foldable_-news-72314.php,17
https://www.gsmarena.com/oppo_find_x9s_pro_design_and_color_options_leaked-news-72308.php,16


,tweets
day,
2026-04-10,4
2026-04-13,3
2026-04-16,3
2026-04-20,3
2026-05-14,3
2026-05-18,3
2026-05-21,3


## Siguiente paso

Con estas tablas ya puedes filtrar, agrupar, unir y limpiar datos con pandas antes de pasar a análisis más específicos o visualizaciones.

## Preparación de tweets para EDA y NLP

En esta sección convertimos los tweets Bronze en una tabla plana, limpiamos el texto, detectamos idioma y dejamos un CSV base completo para el análisis de sentimientos y el EDA.

In [30]:
import re
from pathlib import Path

from langdetect import DetectorFactory, LangDetectException, detect

DetectorFactory.seed = 42


TWEETS_OUTPUT_DIR = PROJECT_ROOT / 'notebooks' / 'output'
TWEETS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TWEETS_CSV_PATH = TWEETS_OUTPUT_DIR / 'bronze_twitter_eda.csv'


snapshot_tweets = bronze_tables['twitter_tweets'].copy()
print(f'Snapshot rows: {len(snapshot_tweets)}')
print(snapshot_tweets.columns.to_list())


def build_twitter_flat_table(snapshot_df: pd.DataFrame) -> pd.DataFrame:
    exploded = snapshot_df.explode('tweets', ignore_index=True)
    snapshot_part = exploded.drop(columns=['tweets']).add_prefix('snapshot_')
    tweet_details = pd.json_normalize(exploded['tweets'], sep='_').add_prefix('tweet_')
    flat = pd.concat([snapshot_part, tweet_details], axis=1)
    return flat


def clean_text(text: str | None) -> str:
    if text is None:
        return ''
    cleaned = str(text)
    cleaned = re.sub(r'https?://\S+|www\.\S+', ' ', cleaned)
    cleaned = re.sub(r'@\w+', ' ', cleaned)
    cleaned = re.sub(r'#', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def detect_language(text: str) -> str:
    if not text:
        return 'unknown'
    try:
        return detect(text)
    except LangDetectException:
        return 'unknown'


tweets_flat = build_twitter_flat_table(snapshot_tweets)
print(f'Flat rows: {len(tweets_flat)}')
tweets_flat.head(3)

Snapshot rows: 22
['_id', 'tweets', 'has_next_page', 'next_cursor', 'createdAt', 'updatedAt', '__v', 'bronze_file']
Flat rows: 269


,snapshot__id,snapshot_has_next_page,snapshot_next_cursor,snapshot_createdAt,snapshot_updatedAt,snapshot___v,snapshot_bronze_file,tweet_type,tweet_id,tweet_url,...,tweet_quoted_tweet_article,tweet_quoted_tweet_author_profile_bio_entities_url_urls,tweet_quoted_tweet_quoted_tweet,tweet_card_binding_values,tweet_card_card_platform_platform_audience_name,tweet_card_card_platform_platform_device_name,tweet_card_card_platform_platform_device_version,tweet_card_name,tweet_card_url,tweet_card_user_refs_results
0,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2026-04-10 03:31:20.837,2026-04-10T03:31:20.837000,0,twitter_tweets_20260411_024745.json,tweet,2042444537281593777,https://x.com/ProfessorCardz/status/2042444537...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2026-04-10 03:31:20.837,2026-04-10T03:31:20.837000,0,twitter_tweets_20260411_024745.json,tweet,2042356352027410907,https://x.com/MarghubR16168/status/20423563520...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2026-04-10 03:31:20.837,2026-04-10T03:31:20.837000,0,twitter_tweets_20260411_024745.json,tweet,2041877436258726386,https://x.com/VerdeSelvans/status/204187743625...,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
lang_summary = (
    tweets_selected[['tweet_lang', 'detected_lang', 'is_english']]
    .assign(tweet_lang=lambda frame: frame['tweet_lang'].fillna('unknown'))
    .groupby(['tweet_lang', 'detected_lang', 'is_english'])
    .size()
    .reset_index(name='rows')
    .sort_values('rows', ascending=False)
)

display(lang_summary)

print(f'Rows before export: {len(tweets_selected)}')
print(f'Rows with English flag: {int(tweets_selected["is_english"].sum())}')
print(f'Rows without English flag: {int((~tweets_selected["is_english"]).sum())}')

tweets_eda = tweets_selected.copy()
tweets_eda.to_csv(TWEETS_CSV_PATH, index=False, encoding='utf-8')
print(f'Saved CSV to {TWEETS_CSV_PATH}')
tweets_eda.head(5)

,tweet_lang,detected_lang,is_english,rows
0,en,en,True,254
2,unknown,hu,False,9
1,in,en,True,6


Rows before export: 269
Rows with English flag: 260
Rows without English flag: 9
Saved CSV to /home/naciscric/Documents/university/2026-1/Data-Analysis-Programming/notebooks/output/bronze_twitter_eda.csv


,snapshot_bronze_file,snapshot__id,snapshot_has_next_page,snapshot_next_cursor,tweet_id,tweet_text,tweet_createdAt,tweet_lang,tweet_source,tweet_retweetCount,...,tweet_author_isVerified,tweet_author_isBlueVerified,tweet_author_location,tweet_author_createdAt,tweet_text_raw,clean_text,text_length,detected_lang,is_english,lang_match
0,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2042444537281593777,Hey iPhone battery replacers! \n\niPhone 13 - ...,Fri Apr 10 03:28:19 +0000 2026,en,Twitter for iPhone,0.0,...,False,True,New Delhi,Sun Jan 08 13:07:44 +0000 2023,Hey iPhone battery replacers! \n\niPhone 13 - ...,Hey iPhone battery replacers! iPhone 13 - 4 ye...,144,en,True,True
1,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2042356352027410907,@SamsungIndia @SamsungSupport\nVery disappoint...,Thu Apr 09 21:37:54 +0000 2026,en,Twitter for iPhone,0.0,...,False,False,,Wed Nov 08 07:22:23 +0000 2023,@SamsungIndia @SamsungSupport\nVery disappoint...,Very disappointed. I bought a Samsung Galaxy S...,238,en,True,True
2,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041877436258726386,After using the Samsung Galaxy S25 FE for over...,Wed Apr 08 13:54:51 +0000 2026,en,Twitter for iPhone,21.0,...,False,True,DIVE,Tue Sep 08 16:28:23 +0000 2020,After using the Samsung Galaxy S25 FE for over...,After using the Samsung Galaxy S25 FE for over...,519,en,True,True
3,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041654754456301988,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,Tue Apr 07 23:10:00 +0000 2026,en,Twitter for iPhone,0.0,...,False,True,"Boulder, CO",Mon Nov 05 17:56:51 +0000 2007,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,The PELADN WO4 AMD Ryzen 5 Mini PC is an amazi...,232,en,True,True
4,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041072528534495498,- Nokia 7210 Supernova (loved the speaker and ...,Mon Apr 06 08:36:26 +0000 2026,en,Twitter for iPhone,0.0,...,False,False,end of line club,Fri Oct 18 06:25:27 +0000 2019,- Nokia 7210 Supernova (loved the speaker and ...,- Nokia 7210 Supernova (loved the speaker and ...,279,en,True,True


In [32]:
tweet_column_audit = pd.DataFrame({
    'column': tweets_flat.columns,
    'dtype': tweets_flat.dtypes.astype(str).values,
    'missing_ratio': tweets_flat.isna().mean().values,
}).sort_values(['missing_ratio', 'column'], ascending=[False, True])

display(tweet_column_audit.head(40))

analysis_columns = [
    'snapshot_bronze_file',
    'snapshot__id',
    'snapshot_has_next_page',
    'snapshot_next_cursor',
    'tweet_id',
    'tweet_text',
    'tweet_createdAt',
    'tweet_lang',
    'tweet_source',
    'tweet_retweetCount',
    'tweet_replyCount',
    'tweet_likeCount',
    'tweet_quoteCount',
    'tweet_viewCount',
    'tweet_isReply',
    'tweet_conversationId',
    'tweet_author_userName',
    'tweet_author_name',
    'tweet_author_id',
    'tweet_author_followers',
    'tweet_author_following',
    'tweet_author_isVerified',
    'tweet_author_isBlueVerified',
    'tweet_author_location',
    'tweet_author_createdAt',
]

analysis_columns = [column for column in analysis_columns if column in tweets_flat.columns]
columns_drop = [column for column in tweets_flat.columns if column not in analysis_columns]

print('Columns keep:', analysis_columns)
print('Columns drop:', columns_drop)

tweets_selected = tweets_flat[analysis_columns].copy()
tweets_selected['tweet_text_raw'] = tweets_selected['tweet_text'].astype('string')
tweets_selected['clean_text'] = tweets_selected['tweet_text_raw'].map(clean_text)
tweets_selected['text_length'] = tweets_selected['clean_text'].str.len()
tweets_selected['detected_lang'] = tweets_selected['clean_text'].fillna('').map(detect_language)
tweets_selected['is_english'] = tweets_selected['tweet_lang'].eq('en') | tweets_selected['detected_lang'].eq('en')
tweets_selected['lang_match'] = tweets_selected['tweet_lang'].fillna('unknown').eq(tweets_selected['detected_lang'].fillna('unknown'))

tweets_selected.head(5)

,column,dtype,missing_ratio
32,tweet_article,float64,1.000000
69,tweet_author_automatedBy,float64,1.000000
41,tweet_author_verifiedType,float64,1.000000
27,tweet_card,float64,1.000000
31,tweet_communityInfo,float64,1.000000
22,tweet_inReplyToId,float64,1.000000
28,tweet_quoted_tweet,float64,1.000000
167,tweet_quoted_tweet_article,float64,1.000000
131,tweet_quoted_tweet_author_automatedBy,float64,1.000000
104,tweet_quoted_tweet_author_verifiedType,float64,1.000000


Columns keep: ['snapshot_bronze_file', 'snapshot__id', 'snapshot_has_next_page', 'snapshot_next_cursor', 'tweet_id', 'tweet_text', 'tweet_createdAt', 'tweet_lang', 'tweet_source', 'tweet_retweetCount', 'tweet_replyCount', 'tweet_likeCount', 'tweet_quoteCount', 'tweet_viewCount', 'tweet_isReply', 'tweet_conversationId', 'tweet_author_userName', 'tweet_author_name', 'tweet_author_id', 'tweet_author_followers', 'tweet_author_following', 'tweet_author_isVerified', 'tweet_author_isBlueVerified', 'tweet_author_location', 'tweet_author_createdAt']
Columns drop: ['snapshot_createdAt', 'snapshot_updatedAt', 'snapshot___v', 'tweet_type', 'tweet_url', 'tweet_twitterUrl', 'tweet_bookmarkCount', 'tweet_inReplyToId', 'tweet_displayTextRange', 'tweet_inReplyToUserId', 'tweet_inReplyToUsername', 'tweet_card', 'tweet_quoted_tweet', 'tweet_retweeted_tweet', 'tweet_isLimitedReply', 'tweet_communityInfo', 'tweet_article', 'tweet_author_type', 'tweet_author_url', 'tweet_author_twitterUrl', 'tweet_author_ve

,snapshot_bronze_file,snapshot__id,snapshot_has_next_page,snapshot_next_cursor,tweet_id,tweet_text,tweet_createdAt,tweet_lang,tweet_source,tweet_retweetCount,...,tweet_author_isVerified,tweet_author_isBlueVerified,tweet_author_location,tweet_author_createdAt,tweet_text_raw,clean_text,text_length,detected_lang,is_english,lang_match
0,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2042444537281593777,Hey iPhone battery replacers! \n\niPhone 13 - ...,Fri Apr 10 03:28:19 +0000 2026,en,Twitter for iPhone,0.0,...,False,True,New Delhi,Sun Jan 08 13:07:44 +0000 2023,Hey iPhone battery replacers! \n\niPhone 13 - ...,Hey iPhone battery replacers! iPhone 13 - 4 ye...,144,en,True,True
1,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2042356352027410907,@SamsungIndia @SamsungSupport\nVery disappoint...,Thu Apr 09 21:37:54 +0000 2026,en,Twitter for iPhone,0.0,...,False,False,,Wed Nov 08 07:22:23 +0000 2023,@SamsungIndia @SamsungSupport\nVery disappoint...,Very disappointed. I bought a Samsung Galaxy S...,238,en,True,True
2,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041877436258726386,After using the Samsung Galaxy S25 FE for over...,Wed Apr 08 13:54:51 +0000 2026,en,Twitter for iPhone,21.0,...,False,True,DIVE,Tue Sep 08 16:28:23 +0000 2020,After using the Samsung Galaxy S25 FE for over...,After using the Samsung Galaxy S25 FE for over...,519,en,True,True
3,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041654754456301988,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,Tue Apr 07 23:10:00 +0000 2026,en,Twitter for iPhone,0.0,...,False,True,"Boulder, CO",Mon Nov 05 17:56:51 +0000 2007,The PELADN WO4 AMD Ryzen 5 Mini PC is an amaz...,The PELADN WO4 AMD Ryzen 5 Mini PC is an amazi...,232,en,True,True
4,twitter_tweets_20260411_024745.json,69d86f08aae727ea3c009a31,True,DAADDAABCgABHFg4fD4bQbEKAAIcSlokVJpQ7AAIAAIAAA...,2041072528534495498,- Nokia 7210 Supernova (loved the speaker and ...,Mon Apr 06 08:36:26 +0000 2026,en,Twitter for iPhone,0.0,...,False,False,end of line club,Fri Oct 18 06:25:27 +0000 2019,- Nokia 7210 Supernova (loved the speaker and ...,- Nokia 7210 Supernova (loved the speaker and ...,279,en,True,True
